# Wrist-Fracture Detector — Colab (free GPU) trainer

Trains the GRAZPEDWRI-DX fracture detector on a free Colab **T4 GPU** and produces the two files the app auto-detects:

- `model/detector/weights/detector.pt`
- `model/detector/weights/detector_meta.json`

**Before you start:** run `prepare_grazpedwri.py` locally (see `docs/TRAINING.md`) to build the patient-level `yolo_fracture/` split, then zip that folder and upload the zip to your Google Drive, e.g. `MyDrive/bfd/yolo_fracture.zip`.

Enable the GPU: **Runtime → Change runtime type → T4 GPU**.

Hyperparameters here mirror `model/detector/train_detector.py` exactly (640 px, no vertical flips, seed 1337), so the result is identical to a local GPU run.

In [ ]:
# 1. GPU check + install
!nvidia-smi -L
!pip -q install ultralytics

In [ ]:
# 2. Mount Drive and unzip the prepared dataset
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/bfd/yolo_fracture.zip'  # <-- adjust if needed
!rm -rf /content/yolo_fracture
!unzip -q "$ZIP" -d /content/
# If the zip contains a top-level 'yolo_fracture/' folder this is already right:
import os
DATA = '/content/yolo_fracture'
assert os.path.isdir(f'{DATA}/images/train'), 'Expected images/train under the unzipped folder'
print('images/train:', len(os.listdir(f'{DATA}/images/train')))

In [ ]:
# 3. Rewrite dataset.yaml 'path:' to the Colab location (it was written with a Windows path)
yaml_path = f'{DATA}/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(
        f'path: {DATA}\n'
        'train: images/train\nval: images/val\ntest: images/test\n'
        'names:\n  0: fracture\n'
    )
print(open(yaml_path).read())

In [ ]:
# 4. Train — same settings as model/detector/train_detector.py
from ultralytics import YOLO

BASE_MODEL = 'yolo11n.pt'   # nano: fast CPU inference in the app
EPOCHS = 100
IMGSZ = 640

model = YOLO(BASE_MODEL)
results = model.train(
    data=f'{DATA}/dataset.yaml',
    epochs=EPOCHS, imgsz=IMGSZ, seed=1337,
    flipud=0.0,   # never flip radiographs vertically
    fliplr=0.5, degrees=5.0, translate=0.08, scale=0.12,
    project='/content/runs', name='grazpedwri_fracture',
)
print('Best weights:', results.save_dir)

In [ ]:
# 5. Validate on the held-out TEST split and write detector_meta.json
import json, shutil, os
from datetime import datetime, timezone

best = os.path.join(str(results.save_dir), 'weights', 'best.pt')
shutil.copyfile(best, '/content/detector.pt')

val = YOLO('/content/detector.pt').val(data=f'{DATA}/dataset.yaml', split='test')
meta = {
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'base_model': BASE_MODEL,
    'dataset': 'GRAZPEDWRI-DX (fracture class only, patient-level split)',
    'domain': 'pediatric wrist radiographs — beta outside wrist studies',
    'imgsz': IMGSZ, 'epochs': EPOCHS, 'confidence_threshold': 0.25,
    'test_metrics': {
        'mAP50': float(val.box.map50), 'mAP50_95': float(val.box.map),
        'precision': float(val.box.mp), 'recall': float(val.box.mr),
    },
    'license_note': 'Ultralytics YOLO is AGPL-3.0 (see docs/TRAINING.md).',
}
with open('/content/detector_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta['test_metrics'], indent=2))

In [ ]:
# 6. Download the two files, then drop them into model/detector/weights/ locally
from google.colab import files
files.download('/content/detector.pt')
files.download('/content/detector_meta.json')

## Back on your machine

1. Drop the two downloaded files here:
```
model/detector/weights/detector.pt
model/detector/weights/detector_meta.json
```

2. Install ultralytics for **local inference** (stop the dev server first — on Windows it locks `cv2.pyd`):
```
pip install -r requirements-train.txt
pip install --force-reinstall opencv-python-headless==5.0.0.93   # ultralytics pulls full opencv; restore headless
```

3. Restart the app. `predict.py` calls `weights_available()`, sees the weights + `ultralytics`, and the pipeline gains a **"localizing"** step that draws boxes in the viewer and PDF (labeled *beta — wrist-validated*). No code change needed. Without either the weights or ultralytics, localization stays cleanly disabled.